In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import csv
import os

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Generator Network
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(2, 16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 2),
        )

    def forward(self, input):
        return self.main(input)

# Discriminator Network
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(2, 16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 1),
        )

    def forward(self, input):
        return self.main(input)

# Initialize models and move to device
generator = Generator().to(device)
discriminator = Discriminator().to(device)

# Loss and optimizers
criterion = nn.BCEWithLogitsLoss()
gen_optimizer = optim.RMSprop(generator.parameters(), lr=0.001)
disc_optimizer = optim.RMSprop(discriminator.parameters(), lr=0.001)

# Noise sampling
def sample_Z(m, n):
    return torch.Tensor(np.random.uniform(-1., 1., size=[m, n])).to(device)

# Load your nuclear flux data (adjust path as needed)
def load_data(file_path):
    data = np.loadtxt(file_path, usecols=[0,1])
    return torch.Tensor(data).to(device)

real_data_file = '/home/jovyan/FluxGAN/code/m_hist.txt'
x_plot = load_data(real_data_file)

# Training params
batch_size = min(256, len(x_plot))
nd_steps = 10
ng_steps = 10
num_iterations = 100001

# Setup directories
log_dir = '../plots'
checkpoint_dir = os.path.join(log_dir, 'checkpoint')
plot_dir = os.path.join(log_dir, 'iterations')
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)

loss_log_file = os.path.join(log_dir, 'loss_log.csv')

# Checkpoint saving
def save_checkpoint(iteration):
    path = os.path.join(checkpoint_dir, f'checkpoint_{iteration}.tar')
    torch.save({
        'iteration': iteration,
        'generator_state_dict': generator.state_dict(),
        'discriminator_state_dict': discriminator.state_dict(),
        'gen_optimizer_state_dict': gen_optimizer.state_dict(),
        'disc_optimizer_state_dict': disc_optimizer.state_dict(),
    }, path)
    print(f"[Checkpoint] Saved at iteration {iteration}")

# Checkpoint loading
def load_checkpoint():
    files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.tar')]
    if not files:
        print("[Checkpoint] No checkpoint found. Starting fresh.")
        return 0
    latest = max(files, key=lambda f: int(f.split('_')[1].split('.')[0]))
    path = os.path.join(checkpoint_dir, latest)
    checkpoint = torch.load(path)
    generator.load_state_dict(checkpoint['generator_state_dict'])
    discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
    gen_optimizer.load_state_dict(checkpoint['gen_optimizer_state_dict'])
    disc_optimizer.load_state_dict(checkpoint['disc_optimizer_state_dict'])
    print(f"[Checkpoint] Loaded from iteration {checkpoint['iteration']}")
    return checkpoint['iteration']

# Load checkpoint if exists
start_iteration = load_checkpoint()

# Initialize CSV log file if fresh start
if start_iteration == 0:
    with open(loss_log_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Iteration', 'Discriminator Loss', 'Generator Loss'])

# Training loop
for i in range(start_iteration, num_iterations):
    for _ in range(nd_steps):
        indices = np.random.choice(len(x_plot), batch_size, replace=False)
        real_batch = x_plot[indices]
        noise_batch = sample_Z(batch_size, 2)

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # Discriminator real loss
        real_out = discriminator(real_batch)
        real_loss = criterion(real_out, real_labels)

        # Discriminator fake loss
        fake_data = generator(noise_batch).detach()
        fake_out = discriminator(fake_data)
        fake_loss = criterion(fake_out, fake_labels)

        disc_loss = real_loss + fake_loss

        disc_optimizer.zero_grad()
        disc_loss.backward()
        disc_optimizer.step()

    for _ in range(ng_steps):
        noise_batch = sample_Z(batch_size, 2)
        fake_data = generator(noise_batch)
        fake_out = discriminator(fake_data)
        gen_loss = criterion(fake_out, real_labels)

        gen_optimizer.zero_grad()
        gen_loss.backward()
        gen_optimizer.step()

    # Log losses
    with open(loss_log_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([i, disc_loss.item(), gen_loss.item()])

    if i % 100 == 0 or i == start_iteration:
        print(f"Iteration {i}: Disc Loss={disc_loss.item():.4f}, Gen Loss={gen_loss.item():.4f}")

    # Save checkpoint every 1000 iterations
    if i % 1000 == 0 and i != start_iteration:
        save_checkpoint(i)

    # Save plot every 1000 iterations
    if i % 1000 == 0:
        with torch.no_grad():
            generated_samples = generator(sample_Z(batch_size, 2)).cpu().numpy()
            real_samples = x_plot.cpu().numpy()

        plt.figure(figsize=(6,6))
        plt.scatter(real_samples[:, 0], real_samples[:, 1], color='blue', alpha=0.6, label='Real Data')
        plt.scatter(generated_samples[:, 0], generated_samples[:, 1], color='orange', alpha=0.6, label='Generated Data')
        plt.legend()
        plt.title(f"GAN Flux Samples at Iteration {i}")
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, f'iteration_{i}.png'), dpi=300)
        plt.close()
